# Medallion Architecture with Spark and Iceberg

## Initialize Spark session

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 2, 
         "executorMemory": "14G",
         "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
         "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.sql.catalog.spark_catalog":"org.apache.iceberg.spark.SparkSessionCatalog",
         "spark.sql.catalog.spark_catalog.type":"glue",
         "spark.hadoop.hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"
    }
}

## Variables and libraries setup

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql.functions import when, col, lit, to_timestamp, regexp_extract, lower
import datetime
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

BRONZE_DB="bronze_db"
SILVER_DB="silver_db"
GOLD_DB="gold_db"
BRONZE_TABLE="orders"
SILVER_TABLE="orders"
GOLD_TABLE="orders_by_year_state_category"

#Change with your bucket
S3_BASE_LOCATION="s3://mybucket"

## Create DBs and tables

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB} LOCATION '{S3_BASE_LOCATION}/{BRONZE_DB}'")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB} LOCATION '{S3_BASE_LOCATION}/{SILVER_DB}'")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB} LOCATION '{S3_BASE_LOCATION}/{GOLD_DB}'")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_DB}.{BRONZE_TABLE} (
    Op STRING,
    dms_received_ts TIMESTAMP,
    order_id STRING,
    customer_id STRING,
    customer_email STRING,
    order_value DOUBLE,
    shipping_address STRING,
    state STRING,
    order_creation_ts TIMESTAMP,
    category INT
) USING iceberg
PARTITIONED BY (day(order_creation_ts))
""")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_DB}.{SILVER_TABLE} (
    order_id STRING,
    customer_id STRING,
    customer_email STRING,
    order_value DOUBLE,
    shipping_address STRING,
    state STRING,
    category STRING,
    order_creation_ts TIMESTAMP
) USING iceberg
PARTITIONED BY (day(order_creation_ts))
""")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_DB}.{GOLD_TABLE} (
    state STRING,
    category STRING,
    orders_total_amount DOUBLE,
    year INT,
    num_orders INT
) USING iceberg
PARTITIONED BY (year)
""")

## Helper functions

In [ ]:
def propagate_bronze_to_silver(verbose=True):
    """
    Propagate data from bronze to silver layer with validations:
    1. order_value not negative (set to 0 if negative)
    2. customer_email matches regex (set to null if invalid)
    3. customer_email converted to lowercase
    4. order_id not null (remove record if null)
    
    Uses incremental processing with snapshot IDs and stores watermark in table properties.
    """
    
    print("============== BRONZE => SILVER ==============")

    try:
        # Get all snapshots Bronze
        snapshots = spark.sql(f"SELECT snapshot_id FROM {BRONZE_DB}.{BRONZE_TABLE}.snapshots ORDER BY committed_at").collect()
        if not snapshots:
            print("No snapshots found in Bronze.")
            return

        #Get latest snapshot in the Bronze table
        latest_snapshot_bronze = str(snapshots[-1]['snapshot_id'])
        
        # Get the last Bronze processed snapshot ID from Silver table properties
        properties_df = spark.sql(f"SHOW TBLPROPERTIES {SILVER_DB}.{SILVER_TABLE}")
        watermark_snapshot = None
        
        try:
            watermark_snapshot = properties_df.filter("key = 'bronze_watermark'").select("value").collect()[0][0]
            print(f"Found bronze_watermark {watermark_snapshot}")
        except:
            print("No watermark found. Processing all data.")
            

        if latest_snapshot_bronze == watermark_snapshot:
            print("No new Bronze snapshot; skipping Bronze replication.")
            return
        
        
        # Read data incrementally from bronze table
        if watermark_snapshot:
            print(f"Processing changes from snapshot {watermark_snapshot} to {latest_snapshot_bronze}")
            bronze_df = spark.read.format("iceberg") \
                .option("start-snapshot-id", watermark_snapshot) \
                .option("end-snapshot-id", latest_snapshot_bronze) \
                .load(f"{BRONZE_DB}.{BRONZE_TABLE}")
        else:
            print(f"Processing all data from bronze table up to {latest_snapshot_bronze}")
            bronze_df = spark.read.option("snapshot-id", latest_snapshot_bronze).format("iceberg").load(f"{BRONZE_DB}.{BRONZE_TABLE}")
        
        # Count records before validation
        records_before = bronze_df.count()
        print(f"Records read from bronze: {records_before}")
        
        if verbose:
            print("Data pulled from Bronze")
            bronze_df.orderBy("order_id","dms_received_ts").show(truncate=False)
        
        # Define window spec for deduplication
        window_spec = Window.partitionBy("order_id").orderBy(col("dms_received_ts").desc())
        
        # Add row number within each order_id partition
        bronze_df_with_row_num = bronze_df.withColumn("row_num", row_number().over(window_spec))
        
        # Keep only the latest record for each order_id
        deduplicated_df = bronze_df_with_row_num.filter(col("row_num") == 1).drop("row_num")
        
        records_after_dedup = deduplicated_df.count()
        print(f"Records after deduplication: {records_after_dedup}")
        print(f"Duplicate records removed: {records_before - records_after_dedup}")
        
        if verbose:
            print("Data after deduplication:")
            deduplicated_df.orderBy("order_id","dms_received_ts").show(truncate=False)
        
        # Apply validations
        validated_df = deduplicated_df \
            .withColumn("order_value", when(col("order_value") < 0, 0).otherwise(col("order_value"))) \
            .withColumn("customer_email", 
                       when(regexp_extract(col("customer_email"), EMAIL_REGEX, 0) == "", None)
                       .otherwise(lower(col("customer_email")))) \
            .withColumn("category", 
                   when(col("category") == 1, "Budget")
                   .when(col("category") == 2, "Standard")
                   .when(col("category") == 3, "Premium")
                   .otherwise(None)) \
            .filter(col("order_id").isNotNull())
        
        if verbose:
            print("Data after sanitization:")
            validated_df.orderBy("order_id","dms_received_ts").show(truncate=False)

        
        # Count records after validation
        records_after = validated_df.count()
        print(f"Records after validation: {records_after}")
        print(f"Records filtered out during validation: {records_after_dedup - records_after}")
        
        # Log validation metrics
        invalid_emails = deduplicated_df.filter(regexp_extract(col("customer_email"), EMAIL_REGEX, 0) == "").count()
        negative_values = deduplicated_df.filter(col("order_value") < 0).count()
        null_order_ids = deduplicated_df.filter(col("order_id").isNull()).count()
        
        print(f"Data quality metrics:")
        print(f"  - Invalid emails: {invalid_emails}")
        print(f"  - Negative order values: {negative_values}")
        print(f"  - Null order IDs: {null_order_ids}")
        
        # Select required columns for silver layer
        silver_df = validated_df.select(
            "Op",
            "order_id", 
            "customer_id", 
            "customer_email", 
            "order_value", 
            "shipping_address", 
            "state", 
            "category",
            "order_creation_ts"
        )
        
        # Register as temp view for merge operation
        silver_df.createOrReplaceTempView("silver_updates")
        
        print("Running MERGE INTO silver")

        # Merge into silver table
        spark.sql(f"""
        MERGE INTO {SILVER_DB}.{SILVER_TABLE} t
        USING silver_updates s
        ON t.order_id = s.order_id
        WHEN MATCHED AND s.Op = 'U' THEN
            UPDATE SET 
                t.customer_id = s.customer_id,
                t.customer_email = s.customer_email,
                t.order_value = s.order_value,
                t.shipping_address = s.shipping_address,
                t.state = s.state,
                t.category = s.category,
                t.order_creation_ts = s.order_creation_ts
        WHEN MATCHED AND s.Op = 'D' THEN
            DELETE
        WHEN NOT MATCHED AND s.Op IN ('I', 'U') THEN
            INSERT (
                order_id,
                customer_id,
                customer_email,
                order_value,
                shipping_address,
                state,
                category,
                order_creation_ts
            ) VALUES (
                s.order_id,
                s.customer_id,
                s.customer_email,
                s.order_value,
                s.shipping_address,
                s.state,
                s.category,
                s.order_creation_ts
            )
        """)
        
        print("Updating bronze_watermark in silver table")
        
        # Update watermark in table properties
        spark.sql(f"""
        ALTER TABLE {SILVER_DB}.{SILVER_TABLE} SET TBLPROPERTIES (
            'bronze_watermark' = '{latest_snapshot_bronze}'
        )
        """)
        
        print(f"Bronze to Silver propagation completed. Processed up to snapshot {latest_snapshot_bronze}")
        return silver_df
    
    except Exception as e:
        print(f"Error in Bronze to Silver propagation: {str(e)}")
        raise

In [ ]:
def propagate_silver_to_gold(verbose=True):
    """
    Propagate data from silver to gold layer for analytical aggregations:
    1. Uses Iceberg's changelog view to identify affected records
    2. Aggregates orders by state, category, and year dimensions
    3. Calculates total order amounts and num orders for each dimension combination
    4. Only recomputes aggregations for affected dimension combinations
    
    Uses incremental processing with snapshot IDs and stores watermark in table properties.
    The function tracks the last processed silver snapshot ID in the gold table properties
    to enable efficient incremental processing in subsequent runs.
    """
    
    print("============== SILVER => GOLD ==============")
        
    try:
        # Get all snapshots in Silver table
        snapshots = spark.sql(f"SELECT snapshot_id FROM {SILVER_DB}.{SILVER_TABLE}.snapshots ORDER BY committed_at").collect()
        if not snapshots:
            print("No snapshots found in Silver.")
            return

        #Get latest snapshot in the Silver table
        latest_snapshot_silver = str(snapshots[-1]['snapshot_id'])
        
        # Get the last Silver processed snapshot ID from Gold table properties
        properties_df = spark.sql(f"SHOW TBLPROPERTIES {GOLD_DB}.{GOLD_TABLE}")
        watermark_snapshot = None
        
        try:
            watermark_snapshot = properties_df.filter("key = 'silver_watermark'").select("value").collect()[0][0]
            print(f"Found silver_watermark {watermark_snapshot}")
        except:
            print("No watermark found. Processing all data.")
            

        if latest_snapshot_silver == watermark_snapshot:
            print("No new Silver snapshot; skipping Silver replication.")
            return
        
        CHANGELOG_VIEW_NAME = f"{SILVER_TABLE}_cdc"
        
        
        # Create changelog view to capture changes
        if watermark_snapshot:
            print(f"Processing changes from snapshot {watermark_snapshot} to {latest_snapshot_silver}")
            spark.sql(f"""
            CALL system.create_changelog_view(
                table => '{SILVER_DB}.{SILVER_TABLE}',
                changelog_view => '{CHANGELOG_VIEW_NAME}',
                identifier_columns => array('order_id'),
                compute_updates => false,
                net_changes => true,
                options => map('start-snapshot-id','{watermark_snapshot}','end-snapshot-id', '{latest_snapshot_silver}')
            )
            """)
        else:
            print("Processing all data from silver table")
            spark.sql(f"""
            CALL system.create_changelog_view(
                table => '{SILVER_DB}.{SILVER_TABLE}',
                changelog_view => '{CHANGELOG_VIEW_NAME}',
                options => map('end-snapshot-id', '{latest_snapshot_silver}')
            )
            """)
        
        if verbose:
            print("Change Log View:")
            spark.sql(f"select * from {CHANGELOG_VIEW_NAME}").show(truncate=False)
        
        # Step 1: Create a temporary view for all affected combinations (including those to be deleted)
        spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW all_affected_combinations AS
        SELECT DISTINCT 
            state, 
            category, 
            YEAR(order_creation_ts) as year
        FROM {CHANGELOG_VIEW_NAME}
        WHERE state IS NOT NULL 
          AND category IS NOT NULL 
          AND order_creation_ts IS NOT NULL
        """)
        
        print("Affected aggregation keys:")
        spark.sql("select * from all_affected_combinations").show(truncate=False)

        # Step 2: Filter silver records that belong to affected combinations
        spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW affected_records AS
        SELECT s.*
        FROM {SILVER_DB}.{SILVER_TABLE} VERSION AS OF {latest_snapshot_silver} s
        JOIN all_affected_combinations a
            ON s.state = a.state 
            AND s.category = a.category 
            AND YEAR(s.order_creation_ts) = a.year
        """)
        
        if verbose:
            print("affected_records:")
            spark.sql("select * from affected_records ORDER BY order_id").show(truncate=False)


        # Step 3: Perform aggregations only on the filtered records
        spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW current_aggregations AS
        SELECT 
            state,
            category,
            YEAR(order_creation_ts) as year,
            SUM(order_value) as orders_total_amount,
            COUNT(*) as num_orders
        FROM affected_records
        GROUP BY state, category, YEAR(order_creation_ts)
        """)
        
        if verbose:
            print("current_aggregations:")
            spark.sql("select * from current_aggregations").show(truncate=False)
        
        # Step 4: Identify combinations that should be deleted
        # These are combinations that appear in all_affected_combinations but not in current_aggregations
        spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW combinations_to_delete AS
        SELECT a.state, a.category, a.year
        FROM all_affected_combinations a
        LEFT JOIN current_aggregations c
            ON a.state = c.state 
            AND a.category = c.category 
            AND a.year = c.year
        WHERE c.state IS NULL
        """)
        
        if verbose:
            print("combinations_to_delete:")
            spark.sql("select * from combinations_to_delete").show(truncate=False)
        
        # Step 5: Create final gold_updates view that includes both updates and deletions
        spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW gold_updates AS
        SELECT 
            state,
            category,
            year,
            orders_total_amount,
            num_orders,
            'UPDATE' as operation_type
        FROM current_aggregations
        UNION ALL
        SELECT 
            state,
            category,
            year,
            NULL as orders_total_amount,
            NULL as num_orders,
            'DELETE' as operation_type
        FROM combinations_to_delete
        """)
        
        if verbose:
            print("gold_updates:")
            spark.sql("select * from gold_updates").show(truncate=False)

        
        # Merge all updates in a single operation
        spark.sql(f"""
        MERGE INTO {GOLD_DB}.{GOLD_TABLE} t
        USING gold_updates s
        ON t.state = s.state AND t.category = s.category AND t.year = s.year
        WHEN MATCHED AND s.operation_type = 'UPDATE' THEN
            UPDATE SET 
                t.orders_total_amount = s.orders_total_amount, 
                t.num_orders = s.num_orders
        WHEN MATCHED AND s.operation_type = 'DELETE' THEN
            DELETE
        WHEN NOT MATCHED THEN
            INSERT (
                state,
                category,
                year,
                orders_total_amount,
                num_orders
            ) VALUES (
                s.state,
                s.category,
                s.year,
                s.orders_total_amount,
                s.num_orders
            )
        """)
        
        # Update watermark in table properties
        spark.sql(f"""
        ALTER TABLE {GOLD_DB}.{GOLD_TABLE} SET TBLPROPERTIES (
            'silver_watermark' = '{latest_snapshot_silver}'
        )
        """)
        
        print(f"Silver to Gold propagation completed. Processed up to snapshot {latest_snapshot_silver}")
        return spark.read.format("iceberg").load(f"{GOLD_DB}.{GOLD_TABLE}")
    
    except Exception as e:
        print(f"Error in Silver to Gold propagation: {str(e)}")
        raise

In [ ]:
def show_all_layers():
    print(f"\n===== {BRONZE_DB}.{BRONZE_TABLE} =====")
    spark.sql(f"SELECT * FROM {BRONZE_DB}.{BRONZE_TABLE} ORDER BY order_id, dms_received_ts").show(truncate=False)
    print(f"\n===== {SILVER_DB}.{SILVER_TABLE} =====")
    spark.sql(f"SELECT * FROM {SILVER_DB}.{SILVER_TABLE} ORDER BY order_id").show(truncate=False)
    print(f"\n===== {GOLD_DB}.{GOLD_TABLE} =====")
    spark.sql(f"SELECT * FROM {GOLD_DB}.{GOLD_TABLE} ORDER BY year, state, category").show(truncate=False)

In [ ]:
def run_medallion_propagation(verbose=True):
    print("======> Running Medallion Architecture Propagation")
    propagate_bronze_to_silver(verbose)
    propagate_silver_to_gold(verbose)
    print("##### SHOW ALL LAYERS #####")
    show_all_layers()

In [ ]:
def ingest_to_bronze(bronze_df_batch):
    bronze_df_batch.writeTo(f"{BRONZE_DB}.{BRONZE_TABLE}").append()

In [ ]:
def prepare_batch(data, schema):
    df = spark.createDataFrame(data, schema)
    df = df \
        .withColumn("dms_received_ts", to_timestamp(col("dms_received_ts"), "yyyy-MM-dd HH:mm:ss.SSSSSS")) \
        .withColumn("order_creation_ts", to_timestamp(col("order_creation_ts"), "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    return df

## Data preparation

Create data for 3 batches

In [ ]:
# Define timestamps for each batch
batch1_dms_ts = "2025-10-11 09:05:12.000000"
batch2_dms_ts = "2025-10-15 10:15:22.000000"
batch3_dms_ts = "2025-10-18 14:22:36.000000"
batch4_dms_ts = "2025-10-21 16:45:30.000000"
batch4_dms_ts_later = "2025-10-21 17:30:45.000000"

# Order creation timestamps
order_ts_1 = "2025-10-10 08:30:00.000000"
order_ts_2 = "2025-10-14 09:45:00.000000"
order_ts_3 = "2025-10-17 11:20:00.000000"
order_ts_4 = "2025-10-20 13:10:00.000000"

# Email validation regex pattern
EMAIL_REGEX = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"

# Define schema for test data with category code
schema = StructType([
    StructField("Op", StringType(), True),
    StructField("dms_received_ts", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_email", StringType(), True),
    StructField("order_value", DoubleType(), True),
    StructField("shipping_address", StringType(), True),
    StructField("state", StringType(), True),
    StructField("order_creation_ts", StringType(), True),
    StructField("category", IntegerType(), True)
])

#######################
# BATCH 1 - ALL INSERTS (Just 4 records, one for each key scenario)
#######################
bronze_test_data_batch1 = [
    # Normal record - CA/Standard (150.75 => category 2)
    ("I", batch1_dms_ts, "ORD-001", "CUST-001", "john.doe@example.com", 150.75, "123 Main St", "CA", order_ts_1, 2),
    
    # Normal record - NY/Budget (89.99 => category 1)
    ("I", batch1_dms_ts, "ORD-002", "CUST-002", "jane.smith@example.com", 89.99, "456 Oak Ave", "NY", order_ts_1, 1),
    
    # Normal record - TX/Premium (245.50 => category 3)
    ("I", batch1_dms_ts, "ORD-003", "CUST-003", "bob.johnson@example.com", 245.50, "789 Pine Rd", "TX", order_ts_1, 3),
    
    # Data quality issue - Invalid email (120.00 => category 2)
    ("I", batch1_dms_ts, "ORD-004", "CUST-004", "invalid-email", 120.00, "101 Elm St", "WA", order_ts_1, 2)
]

# Create and prepare batch 1
bronze_df_batch1 = prepare_batch(bronze_test_data_batch1, schema)

#############################################
# BATCH 2 - INSERTS + UPDATES + DELETES (6 records)
#############################################
bronze_test_data_batch2 = [
    # New insert - TX/Premium (210.50 => category 3)
    ("I", batch2_dms_ts, "ORD-005", "CUST-005", "david.miller@example.com", 210.50, "202 Cedar Ln", "TX", order_ts_2, 3),
    
    # New insert - WA/Standard (120.50 => category 2)
    ("I", batch2_dms_ts, "ORD-006", "CUST-006", "sarah.lee@example.com", 120.50, "303 Birch Dr", "WA", order_ts_2, 2),
    
    # Update price - NY/Budget to NY/Standard (89.99 to 110.99 => category 1 to 2)
    ("U", batch2_dms_ts, "ORD-002", "CUST-002", "jane.smith@example.com", 110.99, "456 Oak Ave", "NY", order_ts_1, 2),
    
    # Update email with mixed case - TX/Premium (category unchanged)
    ("U", batch2_dms_ts, "ORD-003", "CUST-003", "Bob.New@Example.COM", 245.50, "789 Pine Rd", "TX", order_ts_1, 3),
    
    # Delete operation
    ("D", batch2_dms_ts, "ORD-001", "CUST-001", "john.doe@example.com", 150.75, "123 Main St", "CA", order_ts_1, 2),
    
    # Data quality issue - Negative value (should be category 1 but will have NULL value)
    ("I", batch2_dms_ts, "ORD-007", "CUST-007", "mike.brown@example.com", -50.00, "404 Maple Ave", "OR", order_ts_2, 1)
]

# Create and prepare batch 2
bronze_df_batch2 = prepare_batch(bronze_test_data_batch2, schema)

#############################################
# BATCH 3 - INSERTS + UPDATES + DELETES (5 records)
#############################################
bronze_test_data_batch3 = [
    # New insert - CA/Standard (175.25 => category 2)
    ("I", batch3_dms_ts, "ORD-008", "CUST-008", "lisa.garcia@example.com", 175.25, "505 Walnut St", "CA", order_ts_3, 2),
    
    # Update state dimension - TX to CA (category unchanged)
    ("U", batch3_dms_ts, "ORD-005", "CUST-005", "david.miller@example.com", 210.50, "202 Cedar Ln", "CA", order_ts_2, 3),
    
    # Update with invalid email
    ("U", batch3_dms_ts, "ORD-006", "CUST-006", "invalid-format", 120.50, "303 Birch Dr", "WA", order_ts_2, 2),
    
    # Delete operation
    ("D", batch3_dms_ts, "ORD-004", "CUST-004", "invalid-email", 120.00, "101 Elm St", "WA", order_ts_1, 2),
    
    # Re-insert previously deleted record with new values (180.25 => category 2)
    ("I", batch3_dms_ts, "ORD-001", "CUST-001", "john.doe.new@example.com", 180.25, "123 Main St Updated", "CA", order_ts_3, 2)
]

# Create and prepare batch 3
bronze_df_batch3 = prepare_batch(bronze_test_data_batch3, schema)

#############################################
# BATCH 4 - MULTIPLE UPDATES + DIMENSION CHANGES (5 records)
#############################################
bronze_test_data_batch4 = [
    # First update to a record (220.50 => category 3)
    ("U", batch4_dms_ts, "ORD-005", "CUST-005", "david.updated@example.com", 220.50, "202 Cedar Ln", "CA", order_ts_2, 3),
    
    # Second update to same record with later timestamp (230.50 => category 3)
    ("U", batch4_dms_ts_later, "ORD-005", "CUST-005", "david.final@example.com", 230.50, "202 Cedar Ln Updated", "CA", order_ts_2, 3),
    
    # Category change via price update (175.25 to 205.25 => category 2 to 3)
    ("U", batch4_dms_ts, "ORD-008", "CUST-008", "lisa.garcia@example.com", 205.25, "505 Walnut St", "CA", order_ts_3, 3),
    
    # Delete the only record for a state/category combination
    ("D", batch4_dms_ts, "ORD-007", "CUST-007", "mike.brown@example.com", -50.00, "404 Maple Ave", "OR", order_ts_2, 1),
    
    # New insert - NY/Premium (250.75 => category 3)
    ("I", batch4_dms_ts, "ORD-009", "CUST-009", "emma.white@example.com", 250.75, "606 Cherry Ln", "NY", order_ts_4, 3)
]

# Create and prepare batch 4
bronze_df_batch4 = prepare_batch(bronze_test_data_batch4, schema)

## Medallion Architecture - Propagation Flow 1

Run ingestion into Bronze with 2 write operations

In [ ]:
ingest_to_bronze(bronze_df_batch1)

In [ ]:
ingest_to_bronze(bronze_df_batch2)

Propagate to SILVER and BRONZE

In [ ]:
run_medallion_propagation(verbose=False)

## Medallion Architecture - Propagation Flow 2

Run ingestion into Bronze with 2 write operations

In [ ]:
ingest_to_bronze(bronze_df_batch3)

In [ ]:
ingest_to_bronze(bronze_df_batch4)

Propagate to SILVER and BRONZE

In [ ]:
run_medallion_propagation(verbose=False)

## Cleanup

In [ ]:
spark.sql(f"drop table {BRONZE_DB}.{BRONZE_TABLE} PURGE")
spark.sql(f"drop table {SILVER_DB}.{SILVER_TABLE} PURGE")
spark.sql(f"drop table {GOLD_DB}.{GOLD_TABLE} PURGE")